# M5. Физический маятник


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    raise RuntimeError("Запустите notebook из корня M5-physical-pendulum.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import config
from scenarios.modes import damped, free, small_angle_damped
from src import model, theory
from src.shapes import Rod

FIGURES_DIR = PROJECT_ROOT / "docs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_figure(fig, name):
    path = FIGURES_DIR / name
    fig.savefig(path, dpi=130, bbox_inches="tight")
    print(f"График сохранён: {path}")

## Форма маятника и момент инерции


In [ ]:
def rotate(points, angle):
    c, s = np.cos(angle), np.sin(angle)
    return points @ np.array([[c, s], [-s, c]])


def body_outlines(pendulum, angle):
    rotation = angle - pendulum.drawing_angle
    return [
        (part, rotate(np.asarray(part.outline(), dtype=float), rotation))
        for part in pendulum.parts
    ]


def draw_pendulum(ax, pendulum, angle):
    for part, points in body_outlines(pendulum, angle):
        if isinstance(part, Rod):
            ax.plot(points[:, 0], points[:, 1], color="0.3", lw=4)
        else:
            ax.fill(
                points[:, 0],
                points[:, 1],
                color="C0",
                alpha=0.6,
                edgecolor="C0",
            )
    center = rotate(
        np.array([pendulum.center_of_mass]), angle - pendulum.drawing_angle
    )[0]
    ax.plot(*center, "x", color="C3", ms=10, mew=2)
    ax.plot(0, 0, "^", color="k", ms=10)


pendulum = config.PENDULUM
omega0 = theory.small_angle_frequency(
    pendulum.inertia, pendulum.mass, pendulum.com_distance, config.G
)
print(pendulum.describe())
print(f"ω_0 = {omega0:.6f} рад/с, T₀ = {2 * np.pi / omega0:.6f} с")

fig, ax = plt.subplots(figsize=(4.5, 6))
ax.set_aspect("equal")
ax.grid(alpha=0.3)
draw_pendulum(ax, pendulum, angle=0.0)
ax.set_title("Форма маятника (× — центр масс)")
save_figure(fig, "0_shape.png")
fig

## Свободные колебания без трения



In [ ]:
p, g = config.PENDULUM, config.G
amplitude = free.INITIAL_ANGLE
t = np.linspace(0, free.SIMULATION_TIME, 4000)
solution = model.simulate(
    p,
    amplitude,
    free.INITIAL_ANGULAR_VELOCITY,
    (0, free.SIMULATION_TIME),
    g,
    free.VISCOUS_DAMPING,
    config.SOLVER_METHOD,
    config.SOLVER_RTOL,
    config.SOLVER_ATOL,
    t_eval=t,
)
angle, angular_velocity = solution.y
energy = model.total_energy(angle, angular_velocity, p, g)
energy_drift = np.max(np.abs(energy - energy[0])) / energy[0]
linear_angle = amplitude * np.cos(omega0 * t)
exact_period = theory.exact_period(amplitude, p.inertia, p.mass, p.com_distance, g)

print(f"Амплитуда: {np.degrees(amplitude):.1f}°")
print(f"T_0 для малых углов: {2 * np.pi / omega0:.6f} с")
print(f"Точный нелинейный период: {exact_period:.6f} с")
print(f"Максимальный относительный дрейф энергии: {energy_drift:.3e}")

fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
axes[0].plot(t, np.degrees(angle), label="численно")
axes[0].plot(t, np.degrees(linear_angle), "--", label="малые углы")
axes[0].set_ylabel("угол, град")
axes[0].legend()
axes[1].plot(t, angular_velocity)
axes[1].set_ylabel("угловая скорость, рад/с")
axes[2].plot(t, (energy - energy[0]) / energy[0])
axes[2].set_ylabel("(E − E₀) / E₀")
axes[2].set_xlabel("t, с")
fig.suptitle(f"Свободные колебания, A = {np.degrees(amplitude):.0f}°")
fig.tight_layout()
save_figure(fig, "1_free_oscillations.png")
fig

## Зависимость периода от амплитуды


In [ ]:
PERIOD_AMPLITUDES_DEG = (
    1,
    5,
    10,
    20,
    30,
    45,
    60,
    75,
    90,
    105,
    120,
    135,
    150,
    160,
    170,
    175,
    179,
)
PERIODS_TO_MEASURE = 5

amplitudes = np.radians(PERIOD_AMPLITUDES_DEG)
small_period = theory.small_angle_period(p.inertia, p.mass, p.com_distance, g)
measured_periods = np.array(
    [
        model.measure_period(
            p,
            value,
            g,
            PERIODS_TO_MEASURE,
            config.SOLVER_METHOD,
            config.SOLVER_RTOL,
            config.SOLVER_ATOL,
        )
        for value in amplitudes
    ]
)
exact_periods = theory.exact_period(amplitudes, p.inertia, p.mass, p.com_distance, g)
series_periods = theory.period_series(amplitudes, p.inertia, p.mass, p.com_distance, g)
relative_error = np.abs(measured_periods - exact_periods) / exact_periods

print(f"T_0 = {small_period:.6f} с")
print(f"Максимальная относительная ошибка: {relative_error.max():.3e}")
for degrees, measured, exact in zip(
    PERIOD_AMPLITUDES_DEG, measured_periods, exact_periods
):
    print(f"A={degrees:3d}°: Tчисл={measured:.6f} с, Tтеор={exact:.6f} с")

fine_amplitudes = np.radians(np.linspace(0.5, 179.5, 400))
fig, axes = plt.subplots(
    2,
    1,
    figsize=(8, 8),
    sharex=True,
    gridspec_kw={"height_ratios": [3, 1]},
)
axes[0].plot(
    np.degrees(fine_amplitudes),
    theory.exact_period(fine_amplitudes, p.inertia, p.mass, p.com_distance, g)
    / small_period,
    label="точная формула",
)
axes[0].plot(
    np.degrees(fine_amplitudes),
    theory.period_series(fine_amplitudes, p.inertia, p.mass, p.com_distance, g)
    / small_period,
    "--",
    label="ряд по амплитуде",
)
axes[0].axhline(1, color="gray", ls=":", label="малые колебания")
axes[0].plot(
    PERIOD_AMPLITUDES_DEG,
    measured_periods / small_period,
    "o",
    color="C3",
    label="численно",
)
axes[0].set_ylabel("T / T₀")
axes[0].legend()
axes[0].set_title("Период в зависимости от амплитуды")
axes[1].semilogy(PERIOD_AMPLITUDES_DEG, relative_error, "o-", color="C3")
axes[1].set_ylabel("отн. ошибка")
axes[1].set_xlabel("амплитуда, град")
fig.tight_layout()
save_figure(fig, "2_period_vs_amplitude.png")
fig

## Сохранение энергии


In [ ]:
t_end = 50 * small_period
t_energy = np.linspace(0, t_end, 20000)
solver_variants = [
    ("RK45", 1e-6, 1e-9),
    ("RK45", 1e-9, 1e-12),
    ("DOP853", 1e-8, 1e-10),
    (config.SOLVER_METHOD, config.SOLVER_RTOL, config.SOLVER_ATOL),
]

fig, ax = plt.subplots(figsize=(9, 5))
for method, rtol, atol in solver_variants:
    energy_solution = model.simulate(
        p,
        free.INITIAL_ANGLE,
        free.INITIAL_ANGULAR_VELOCITY,
        (0, t_end),
        g,
        free.VISCOUS_DAMPING,
        method,
        rtol,
        atol,
        t_eval=t_energy,
    )
    total_energy = model.total_energy(*energy_solution.y, p, g)
    relative_drift = np.abs(total_energy - total_energy[0]) / total_energy[0]
    label = f"{method}, rtol={rtol:.0e}, nfev={energy_solution.nfev}"
    print(f"{label}: max |ΔE|/E = {relative_drift.max():.3e}")
    ax.semilogy(t_energy, np.maximum(relative_drift, 1e-17), label=label)

ax.set_xlabel("t, с")
ax.set_ylabel("|E − E₀| / E₀")
ax.set_title("Дрейф полной энергии за 50 периодов")
ax.legend()
save_figure(fig, "3_energy_conservation.png")
fig

## Затухающие колебания с вязким трением


In [ ]:
damping = damped.VISCOUS_DAMPING
gamma = theory.viscous_decay_rate(damping, p.inertia)
t_damped = np.linspace(0, damped.SIMULATION_TIME, 12000)
small_amplitude = small_angle_damped.INITIAL_ANGLE

small_solution = model.simulate_with_turning_points(
    p,
    small_amplitude,
    small_angle_damped.INITIAL_ANGULAR_VELOCITY,
    (0, small_angle_damped.SIMULATION_TIME),
    g,
    small_angle_damped.VISCOUS_DAMPING,
    config.SOLVER_METHOD,
    config.SOLVER_RTOL,
    config.SOLVER_ATOL,
    t_eval=t_damped,
)
analytic_solution = theory.damped_linear_solution(
    t_damped, small_amplitude, omega0, gamma
)
small_turning_times, small_turning_angles = model.turning_points(small_solution)
gamma_numerical = model.fit_decay_rate(small_turning_times, small_turning_angles)

large_amplitude = damped.INITIAL_ANGLE
large_solution = model.simulate_with_turning_points(
    p,
    large_amplitude,
    damped.INITIAL_ANGULAR_VELOCITY,
    (0, damped.SIMULATION_TIME),
    g,
    damped.VISCOUS_DAMPING,
    config.SOLVER_METHOD,
    config.SOLVER_RTOL,
    config.SOLVER_ATOL,
    t_eval=t_damped,
)
large_turning_times, large_turning_angles = model.turning_points(large_solution)

print(f"γ теоретическое = {gamma:.6f} 1/с")
print(f"γ численное для 2° = {gamma_numerical:.6f} 1/с")
print(f"Q = {theory.quality_factor(omega0, gamma):.1f}")
print(
    "Максимальное отклонение от линейной теории: "
    f"{np.max(np.abs(small_solution.y[0] - analytic_solution)) / small_amplitude:.3e}"
)

fig, axes = plt.subplots(2, 1, figsize=(9, 8))
axes[0].plot(t_damped, np.degrees(small_solution.y[0]), label="численно")
axes[0].plot(t_damped, np.degrees(analytic_solution), "--", label="теория")
axes[0].plot(
    t_damped,
    np.degrees(small_amplitude * np.exp(-gamma * t_damped)),
    "k:",
    label="экспоненциальная огибающая",
)
axes[0].set_title("Вязкое трение, A₀ = 2°")
axes[0].set_ylabel("угол, град")
axes[0].legend()

axes[1].semilogy(
    large_turning_times,
    np.degrees(np.abs(large_turning_angles)),
    ".",
    label="точки поворота",
)
axes[1].semilogy(
    t_damped,
    np.degrees(large_amplitude * np.exp(-gamma * t_damped)),
    "k:",
    label="A₀ exp(−γt)",
)
axes[1].set_title(f"Вязкое трение, A₀ = {np.degrees(large_amplitude):.0f}°")
axes[1].set_xlabel("t, с")
axes[1].set_ylabel("амплитуда, град")
axes[1].legend()
fig.tight_layout()
save_figure(fig, "4_damping.png")
fig